# 01 · Ingestion: from 497 files to 10,943 chunks
This notebook only **inspects** the outputs of `ragtraining.ingest` (discover → extract → chunk). See LOGBOOK D10–D12 and PROBLEMS_LOG P-04…P-22.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 90)
m = pd.read_csv('../data/manifest.csv')
pages = pd.read_parquet('../data/pages.parquet')
chunks = {s: pd.read_parquet(f'../data/chunks_{s}.parquet') for s in ['page','fixed','recursive','hybrid']}

## 1. What was kept and why (manifest)
Every file has a `doc_type`, a `keep` decision and the `reason` (the rule that fired).

In [ ]:
m.pivot_table(index='doc_type', columns='keep', values='path', aggfunc='count', fill_value=0)

In [ ]:
m[m.keep=='no'].groupby('reason').size().sort_values(ascending=False).head(15)

## 2. Where the text comes from
Textbooks dominate by tokens. This is why Phase 3 needs **source balancing**.

In [ ]:
by = pages.groupby('doc_type').n_tokens_est.sum().sort_values()
(by/by.sum()*100).round(1).plot.barh(title='% of tokens by doc_type'); plt.xlabel('%'); plt.show()

In [ ]:
pages.groupby('course').n_tokens_est.sum().sort_values().plot.barh(figsize=(7,6), title='Tokens per course'); plt.show()

## 3. Chunking strategies compared (D12 = hybrid)

In [ ]:
from ragtraining.ingest.chunk import stats
pd.DataFrame({s: stats(df) for s, df in chunks.items()}).T

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 3), sharey=True)
for a, (s, df) in zip(ax, chunks.items()):
    df.n_tokens.clip(upper=800).hist(bins=40, ax=a); a.set_title(s); a.set_xlabel('tokens (est.)')
plt.show()

## 4. Same material, four ways
Financial Intermediation slides p13–17 (tiny slides) and the NLP book's cosine page (p111).

In [ ]:
def show(s, pat, p0, p1):
    d = chunks[s]; d = d[d.path.str.contains(pat) & (d.page_end >= p0) & (d.page_start <= p1)]
    return d[['page_start','page_end','n_tokens','text']].assign(text=d.text.str[:80])
for s in chunks: print(f'=== {s}'); display(show(s, 'your_file_name', 13, 17))

In [ ]:
for s in chunks: print(f'=== {s}'); display(show(s, 'ed3book', 111, 111))

## 5. OCR results (D11)
Scanned PDFs: only lines/pages that passed the quality gate were kept.

In [ ]:
pages[pages.ocr].groupby('path').size()

In [ ]:
print(pages[pages.ocr & pages.path.str.contains('Topic02')].text.iloc[5][:500])